In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
PATH = "/kaggle/input/competitions/ieee-fraud-detection"
id = pd.read_csv(PATH+"/train_identity.csv")
tr = pd.read_csv(PATH+"/train_transaction.csv")

In [3]:
!pip install mlflow dagshub xgboost -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 67.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
import mlflow.sklearn
import mlflow
import dagshub

In [5]:
TARGET = "isFraud"
TR_MISS_TH = 0.8
ID_MISS_TH = 0.7
CARDINALITY_TH = 20
CORRELATION_TH = 0.85

In [ ]:
dagshub.init(repo_owner='lukaLomadze', repo_name='ML_Assignment_2', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow")
mlflow.set_experiment('XGBoost_better')

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=0256bce5-cf1f-48bf-a0a7-10642cfed597&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=1365f3b1f4bda579065efedce779f68f519b2806e370434726dd6c532570c578




Accessing as lukaLomadze

Initialized MLflow to track repo "lukaLomadze/ML_Assignment_2"

Repository lukaLomadze/ML_Assignment_2 initialized!

2026/05/04 15:41:45 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/e24e4485bccb4f5791d4a2778e1efb4a', creation_time=1777909305575, experiment_id='5', last_update_time=1777909305575, lifecycle_stage='active', name='XGBoost', tags={}, trace_location=None, workspace='default'>

# Cleaning

In [7]:
class DropHighMissing(BaseEstimator, TransformerMixin):
    def __init__(self, tr_th=TR_MISS_TH, id_th=ID_MISS_TH, identity_cols=None):
        self.tr_th = tr_th
        self.id_th = id_th
        self.identity_cols = identity_cols

    def fit(self, X, y=None):
        id_set = set(self.identity_cols) if self.identity_cols else set()
        keep = []
        for col in X.columns:
            miss_rate = X[col].isnull().mean()
            threshold = self.id_th if col in id_set else self.tr_th
            if miss_rate <= threshold:
                keep.append(col)
        self.cols_ = keep
        return self

    def transform(self, X):
        return X[self.cols_]

In [ ]:
train = tr.merge(id, on="TransactionID", how="left")
y = train[TARGET].copy()

print(f"Train shape: {train.shape}")
print(f"Fraud rate: {y.mean()}")

Train shape: (590540, 434)
Fraud rate: 0.0350


In [9]:
cat_cols = train.select_dtypes(include='object').columns.tolist()
num_cols = train.select_dtypes(exclude='object').columns.tolist()

In [10]:
with mlflow.start_run(run_name="xgboost_cleaning"):
    mlflow.log_param("TR_MISS_TH", TR_MISS_TH)
    mlflow.log_param("ID_MISS_TH", ID_MISS_TH)
    mlflow.log_param("Final shape", f"{train.shape[0]} X {train.shape[1]}")
    mlflow.log_param("Num cols", len(num_cols))
    mlflow.log_param("Cat cols", len(cat_cols))
    mlflow.log_param("Fraud rate", float(y.mean()))

🏃 View run xgboost_cleaning at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5/runs/6c761e28e2964564a2e503ef228ab066
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5


# Feature Engineering

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, identity_cols=None, identity_ids=None, target=None):
        self.identity_cols = identity_cols
        self.identity_ids = identity_ids
        self.target = target

    def fit(self, X, y=None):
        uid_cols = ["card1", "addr1"]
        if all(c in X.columns for c in uid_cols) and "TransactionAmt" in X.columns:
            uid = X["card1"].astype(str) + "_" + X["addr1"].astype(str)
            tmp = X["TransactionAmt"].copy()
            tmp.index = uid
            self.uid_stats_ = tmp.groupby(level=0).agg(
                uid_tx_count="count",
                uid_amt_mean="mean",
                uid_amt_std="std"
            )
        else:
            self.uid_stats_ = None

        freq_cols = ["card1", "card2", "card3", "card5", "addr1", "addr2"]
        self.freq_maps_ = {}
        for col in freq_cols:
            if col in X.columns:
                self.freq_maps_[col] = X[col].astype(str).value_counts(normalize=True)

        return self

    def transform(self, X):
        X = X.copy()

        if "TransactionAmt" in X.columns:
            X["TransactionAmt_log"]   = np.log1p(X["TransactionAmt"])
            X["TransactionAmt_round"] = np.round(X["TransactionAmt"]).astype("float32")

        if "TransactionDT" in X.columns:
            X["TransactionHour"]  = (X["TransactionDT"] // 3600) % 24
            X["TransactionDay"]   = (X["TransactionDT"] // 86400) % 7
            X["TransactionWeek"]  = (X["TransactionDT"] // 86400) // 7
            X["TransactionMonth"] = (X["TransactionDT"] // 86400) // 30

        if self.uid_stats_ is not None:
            uid = X["card1"].astype(str) + "_" + X["addr1"].astype(str)
            X["uid_tx_count"] = uid.map(self.uid_stats_["uid_tx_count"]).astype("float32")
            X["uid_amt_mean"] = uid.map(self.uid_stats_["uid_amt_mean"]).astype("float32")
            X["uid_amt_std"]  = uid.map(self.uid_stats_["uid_amt_std"]).astype("float32")

        for col, freq_map in self.freq_maps_.items():
            if col in X.columns:
                X[f"{col}_freq"] = X[col].astype(str).map(freq_map).fillna(0).astype("float32")

       
        if self.identity_ids is not None and "TransactionID" in X.columns:
            X["has_identity"] = X["TransactionID"].isin(self.identity_ids).astype(int)

        drop_cols = []
        if "TransactionID" in X.columns:
            drop_cols.append("TransactionID")
        if self.target and self.target in X.columns:
            drop_cols.append(self.target)
        if drop_cols:
            X = X.drop(columns=drop_cols)

        return X

In [12]:
low_card_cols = [c for c in cat_cols if train[c].nunique() <= CARDINALITY_TH]
high_card_cols = [c for c in cat_cols if train[c].nunique() > CARDINALITY_TH]

print(f"Low cardinality cols: {len(low_card_cols)}")
print(f"High cardinality cols: {len(high_card_cols)}")

Low cardinality cols: 25
High cardinality cols: 6


In [ ]:
with mlflow.start_run(run_name="xgboost_feature_engineering"):
    mlflow.log_param("New features", "TransactionDay, TransactionHour, TransactionWeek, TransactionMonth, TransactionAmt_log, TransactionAmt_round, has_identity, uid_tx_count, uid_amt_mean, uid_amt_std, card*_freq, addr*_freq")
    mlflow.log_param("CARDINALITY_TH", CARDINALITY_TH)
    mlflow.log_param("Low cardinality cols", len(low_card_cols))
    mlflow.log_param("High cardinality cols", len(high_card_cols))

🏃 View run xgboost_feature_engineering at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5/runs/c8683fbf65484b84ac88578184560efd
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5


# Feature Selection

In [14]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, th=CORRELATION_TH):
        self.th = th

    def fit(self, X, y=None):
        num = X.select_dtypes(exclude="object")
        if num.shape[1] > 0:
            corr = num.fillna(num.median()).corr().abs()
            upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            self.drop_cols_ = [c for c in upper.columns if any(upper[c] > self.th)]
        else:
            self.drop_cols_ = []
        return self

    def transform(self, X):
        return X.drop(columns=self.drop_cols_, errors="ignore")

In [ ]:
# with mlflow.start_run(run_name="xgboost_feature_selection"):
#     mlflow.log_param("Method", "correlation_filter")
#     mlflow.log_param("CORRELATION_TH", CORRELATION_TH)

🏃 View run xgboost_feature_selection at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5/runs/e62d11588de0482883d2416d3e715778
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5


# Preprocessor

In [ ]:
class Preprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, cardinality_th=20):
        self.cardinality_th = cardinality_th

    def _build(self, X):
        cat_cols  = X.select_dtypes(include="object").columns.tolist()
        low_card  = [c for c in cat_cols if X[c].nunique() <= self.cardinality_th]
        high_card = [c for c in cat_cols if X[c].nunique() >  self.cardinality_th]

        ohe_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
        
        ord_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ])

        
        transformers = []
        if low_card:
            transformers.append(("ohe", ohe_pipe, low_card))
        if high_card:
            transformers.append(("ord", ord_pipe, high_card))

        return ColumnTransformer(transformers, remainder="passthrough")

    def fit(self, X, y=None):
        self.ct_ = self._build(X)
        self.ct_.fit(X, y)
        return self

    def transform(self, X):
        return self.ct_.transform(X)

In [ ]:
def build_pipeline(n_estimators, max_depth, learning_rate, subsample,
                   colsample_bytree, min_child_weight, gamma,
                   scale_pos_weight, colsample_bylevel=1.0):
 

    identity_cols = [c for c in id.columns if c != "TransactionID"]
    identity_ids  = set(id["TransactionID"].unique())

    xgb_params = {
        "n_estimators":      n_estimators,
        "max_depth":         max_depth,
        "learning_rate":     learning_rate,
        "subsample":         subsample,
        "colsample_bytree":  colsample_bytree,
        "colsample_bylevel": colsample_bylevel,
        "min_child_weight":  min_child_weight,
        "gamma":             gamma,
        "scale_pos_weight":  scale_pos_weight,
        "eval_metric":       "auc",
        "tree_method":       "hist",   
        "random_state":      42,
        "n_jobs":            -1,
    }

    return Pipeline([
        ("drop",  DropHighMissing(tr_th=TR_MISS_TH, id_th=ID_MISS_TH, identity_cols=identity_cols)),
        ("feat",  FeatureEngineer(identity_cols=identity_cols, identity_ids=identity_ids, target=TARGET)),
        ("prep",  Preprocessor(cardinality_th=CARDINALITY_TH)),
        ("model", xgb.XGBClassifier(**xgb_params))
    ])

# Training

In [ ]:
scale_pos_weight_value = float((y == 0).sum() / (y == 1).sum())

In [ ]:
spw = scale_pos_weight_value

PARAM_GRID = [
    # baseline
    dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, colsample_bylevel=1.0, min_child_weight=1,  gamma=0,   scale_pos_weight=spw),
    # deeper trees
    dict(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, colsample_bylevel=1.0, min_child_weight=1,  gamma=0,   scale_pos_weight=spw),
    # more trees + lower lr
    dict(n_estimators=500, max_depth=4, learning_rate=0.02, subsample=0.8, colsample_bytree=0.8, colsample_bylevel=1.0, min_child_weight=1,  gamma=0,   scale_pos_weight=spw),
    # stronger regularisation via min_child_weight
    dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, colsample_bylevel=1.0, min_child_weight=10, gamma=0,   scale_pos_weight=spw),
    # gamma regularisation
    dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, colsample_bylevel=1.0, min_child_weight=1,  gamma=1,   scale_pos_weight=spw),
    # lower colsample to reduce overfitting
    dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.7, colsample_bytree=0.6, colsample_bylevel=0.8, min_child_weight=5,  gamma=0.5, scale_pos_weight=spw),
    # high-tree run — best expected AUC
    dict(n_estimators=700, max_depth=5, learning_rate=0.02, subsample=0.8, colsample_bytree=0.7, colsample_bylevel=1.0, min_child_weight=5,  gamma=0,   scale_pos_weight=spw),
]

In [ ]:

CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

best_auc    = -1
best_params = None
results     = []

X = train

In [ ]:
for p in PARAM_GRID:
    with mlflow.start_run(run_name=f"XGB_n={p['n_estimators']}_d={p['max_depth']}_lr={p['learning_rate']}_mcw={p['min_child_weight']}_g={p['gamma']}"):

        pipe = build_pipeline(
            n_estimators      = p["n_estimators"],
            max_depth         = p["max_depth"],
            learning_rate     = p["learning_rate"],
            subsample         = p["subsample"],
            colsample_bytree  = p["colsample_bytree"],
            colsample_bylevel = p.get("colsample_bylevel", 1.0),
            min_child_weight  = p["min_child_weight"],
            gamma             = p["gamma"],
            scale_pos_weight  = p["scale_pos_weight"],
        )

        scores   = cross_val_score(pipe, X, y, cv=CV, scoring="roc_auc", n_jobs=1)
        mean_auc = scores.mean()
        std_auc  = scores.std()

        mlflow.log_params(p)
        mlflow.log_metric("cv_auc_mean", mean_auc)
        mlflow.log_metric("cv_auc_std",  std_auc)

        results.append((p, mean_auc))

        if mean_auc > best_auc:
            best_auc    = mean_auc
            best_params = p

        print(f"AUC: {mean_auc} (+/- {std_auc}) | {p}")

Params: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'sampling': 'smote'} | AUC: 0.8643 (+/- 0.0035)
🏃 View run XGB_n=100_depth=3_lr=0.1_sample=smote at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5/runs/cd5940d6170c402d994feeec1dcb3720
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5
Params: {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'sampling': 'smote'} | AUC: 0.8761 (+/- 0.0031)
🏃 View run XGB_n=100_depth=4_lr=0.1_sample=smote at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5/runs/b5513a85f82241b5a55655764d786b48
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5
Params: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'sampling': 'smote'} | AUC: 0.8859 (+/- 0.0029)
🏃 View run XG

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Params: {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'sampling': 'smote'} | AUC: 0.8850 (+/- 0.0029)
🏃 View run XGB_n=150_depth=4_lr=0.1_sample=smote at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5/runs/dbd253c344d7407ba490f94eb5275a3f
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5
Params: {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'sampling': 'smote'} | AUC: 0.8609 (+/- 0.0024)
🏃 View run XGB_n=100_depth=4_lr=0.05_sample=smote at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5/runs/7eade01e51bf4d4c96c0ac041a817484
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5
Params: {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.2, 'subsample': 0.8, 'colsample_bytree': 0.8, 'sampling': 'smote'} | AUC: 0.8902 (+/- 0.0035)
🏃 View run 

In [ ]:
print(f"\nBest params: {best_params}  |  CV AUC: {best_auc}")

best_pipe = build_pipeline(
    n_estimators      = best_params["n_estimators"],
    max_depth         = best_params["max_depth"],
    learning_rate     = best_params["learning_rate"],
    subsample         = best_params["subsample"],
    colsample_bytree  = best_params["colsample_bytree"],
    colsample_bylevel = best_params.get("colsample_bylevel", 1.0),
    min_child_weight  = best_params["min_child_weight"],
    gamma             = best_params["gamma"],
    scale_pos_weight  = best_params["scale_pos_weight"],
)

with mlflow.start_run(run_name="XGB_better_best_registered") as run:
    mlflow.log_params(best_params)
    mlflow.log_metric("cv_auc_mean", best_auc)

    best_pipe.fit(X, y)

    model_info = mlflow.sklearn.log_model(
        sk_model              = best_pipe,
        artifact_path         = "model",
        registered_model_name = "XGB_better_Best",
    )

    print(f"Model registered: {model_info.model_uri}")
    print(f"Run ID: {run.info.run_id}")


Best params: {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'sampling': 'none', 'scale_pos_weight': np.float64(27.579586700866283)}  |  CV AUC: 0.8995


2026/05/04 16:48:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 16:48:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGB_Best'.
2026/05/04 16:49:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_Best, version 1
Created version '1' of model 'XGB_Best'.


Model registered: models:/m-a5bb6986c52646e59bf25b072ee4f4c6
Run ID: 8361def49427467b907a6d9bdd89b02d
🏃 View run XGB_best_registered at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5/runs/8361def49427467b907a6d9bdd89b02d
🧪 View experiment at: https://dagshub.com/lukaLomadze/ML_Assignment_2.mlflow/#/experiments/5
